# Grokking Deep Learning

My notes and exercises while reading
Grokking Deep Learning by Andrew Trask.

## Chapter 3: Intro to Neural Prediction

### A Simple Neural Network: One Input, One Weight

A neural network makes a prediction by applying a
**weight** to an input. Here, the network has only
one input and one weight.

$$
\text{prediction} = \text{input} \times \text{weight}
$$

The input is the first value in the `number_of_toes`
list (`8.5`), and the weight is `0.1`.

$$
8.5 \times 0.1 = 0.85
$$

The weight determines how strongly the input
influences the prediction.

At this stage, the network is **not learning**:
its weight is fixed, and there is no mechanism
to calculate an error or improve the prediction.

**Key idea:** Learning will involve adjusting
weights to make better predictions.

In [12]:
# Simple neural network example

weight = 0.1
def neural_network(input, weight):
    prediction = input * weight
    return prediction

number_of_toes = [8.5, 9.5, 10, 9]
input = number_of_toes[0]
pred = neural_network(input, weight)
print(pred)

0.8500000000000001


### Neural Network with Multiple Inputs

A neural network can combine several input features
to make one prediction. Each input has an associated
**weight**, which determines its contribution.

The prediction is a **weighted sum**, also called
a dot product:

$$
\text{prediction} = x_1w_1 + x_2w_2 + x_3w_3
$$

In this example:

$$
(8.5 \times 0.1) + (0.65 \times 0.2)
+ (1.2 \times 0) = 0.98
$$

#### Two implementations

- **Pure Python:** `w_sum()` loops through the inputs
  and weights, multiplies each pair, and adds the
  results.
- **NumPy:** `input.dot(weights)` performs the same
  calculation using a vector operation.

#### Key observations

- Each input has its own weight.
- A weight of zero means an input contributes nothing.
- The network combines multiple features into one
  output, but it is still not learning.
- Both implementations produce `0.98`.

In [13]:
# Multiple inputs

def w_sum(a, b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += a[i] * b[i]
    return output

weights = [0.1, 0.2, 0]

def neural_network(input, weights):
    pred = w_sum(input, weights)
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

input = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input, weights)
print(pred)

0.9800000000000001


In [14]:
# Numpy code 

import numpy as np
weights = np.array([0.1, 0.2, 0])
def neural_network(input, weights):
    pred = input.dot(weights)
    return pred
toes = np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])
input = np.array([toes[0], wlrec[0], nfans[0]])
pred = neural_network(input, weights)
print(pred)

0.9800000000000001


## Predicting with Multiple Inputs and Outputs

Previously, our neural network combined three inputs
to produce **one prediction**. Now it uses the same
three inputs to produce **three different predictions**.

Each output has its own set of weights. The network
calculates a separate **weighted sum (dot product)**
for each output.

### Inputs and weights

Our input vector is:

$$
\mathbf{x} = [8.5,\ 0.65,\ 1.2]
$$

The weight matrix contains three rows, one for
each prediction:

$$
W = \begin{bmatrix}
0.1 & 0.1 & -0.3 \\
0.1 & 0.2 & 0.0 \\
0.0 & 1.3 & 0.1
\end{bmatrix}
$$

### Calculating the predictions

Each row of the matrix is multiplied by the
input vector:

$$
\begin{aligned}
\text{Hurt} &= 8.5(0.1) + 0.65(0.1) + 1.2(-0.3)
= 0.555 \\
\text{Win} &= 8.5(0.1) + 0.65(0.2) + 1.2(0)
= 0.980 \\
\text{Sad} &= 8.5(0) + 0.65(1.3) + 1.2(0.1)
= 0.965
\end{aligned}
$$

The network returns a vector of three predictions:

$$
\mathbf{y} = W\mathbf{x}
= [0.555,\ 0.980,\ 0.965]
$$

### Understanding the Python functions

- `w_sum(a, b)` calculates the dot product of
  two vectors.
- `vect_mat_mul(vect, matrix)` calls `w_sum()`
  once for each row of the weight matrix.
- `neural_network(input, weights)` returns
  the resulting predictions.

**Key idea:** Multiple outputs do not require
a fundamentally different calculation. We simply
repeat the weighted sum using different weights
for each output.

With three inputs and three outputs, the network
needs nine weights.

The weights are still fixed: the network is
making predictions, not yet learning.

In [15]:
# Predicting with multiple inputs and outputs

# Calculate the weighted sum (dot product)
def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += a[i] * b[i]
    return output


# Calculate one prediction for each row of weights
def vect_mat_mul(vect, matrix):
    assert len(vect) == len(matrix[0])
    output = [0] * len(matrix)

    for i in range(len(matrix)):
        output[i] = w_sum(vect, matrix[i])

    return output


# Neural network
def neural_network(input, weights):
    pred = vect_mat_mul(input, weights)
    return pred


# Weights for the three predictions: hurt, win, sad
weights = [
    [0.1, 0.1, -0.3],
    [0.1, 0.2, 0.0],
    [0.0, 1.3, 0.1]
]

# Input data
toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

# Select inputs for the first game
input = [toes[0], wlrec[0], nfans[0]]

# Make predictions
pred = neural_network(input, weights)

print("Hurt:", round(pred[0], 3))
print("Win:", round(pred[1], 3))
print("Sad:", round(pred[2], 3))

Hurt: 0.555
Win: 0.98
Sad: 0.965


## Predicting on Predictions: Introducing a Hidden Layer

A neural network can contain multiple layers.
Instead of predicting directly from the inputs,
the network first calculates **intermediate values**,
called the hidden layer.

These values become the inputs to the next layer.

**Architecture:** 3 inputs → 3 hidden neurons → 3 outputs

### How it works

1. Multiply the input vector by the first weight
   matrix to calculate the hidden layer.
2. Multiply the hidden-layer vector by the second
   weight matrix to calculate the final predictions.

Using the book's row-oriented weight matrices:

$$
\mathbf{h} = W_1\mathbf{x}
$$

$$
\mathbf{y} = W_2\mathbf{h}
$$

Therefore:

$$
\mathbf{y} = W_2(W_1\mathbf{x})
$$

### Understanding the code

- `weights[0]` contains the connections from the
  inputs to the hidden layer.
- `weights[1]` contains the connections from the
  hidden layer to the outputs.
- `vect_mat_mul()` is called twice, once for each layer.

**Key idea:** The hidden layer calculates intermediate
values that are used to produce the final predictions.

**Important limitation:** Without nonlinear activation
functions, these two layers are mathematically
equivalent to a single linear transformation.

In [16]:
# Predicting on predictions: a hidden layer

# Input-to-hidden weights
ih_wgt = [
    [0.1, 0.2, -0.1],
    [-0.1, 0.1, 0.9],
    [0.1, 0.4, 0.1]
]

# Hidden-to-output weights
hp_wgt = [
    [0.3, 1.1, -0.3],
    [0.1, 0.2, 0.0],
    [0.0, 1.3, 0.1]
]

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):
    hidden = vect_mat_mul(input, weights[0])
    pred = vect_mat_mul(hidden, weights[1])
    return pred

input = [8.5, 0.65, 1.2]

pred = neural_network(input, weights)
print(pred)

[0.21350000000000002, 0.14500000000000002, 0.5065]
